## **1 кейс**

**Работа с логами**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [1]:
!wget https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log

--2026-06-18 20:50:25--  https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.github.com (gist.github.com)... 198.18.0.6
Connecting to gist.github.com (gist.github.com)|198.18.0.6|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log [following]
--2026-06-18 20:50:28--  https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 198.18.0.7
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|198.18.0.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 459418 (449K) [text/plain]
Saving to: ‘auto_purchase.log’

auto_purchase.log   100%[===================>] 448.65K   555KB/s    in 0.8s    

2026-06-18 20:50:31 (555 KB/s) - ‘auto_purchase.log’ saved [459418/459418]



Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [4]:
with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

for line in lines[400:405]:
    print(line)

INFO  | 2022-11-26 00:01:03,024  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-26, количество людей с автопродлением подписки: 0

INFO  | 2022-11-26 00:01:03,027  | file: demon_auto_purchase_bundle.py |  line: 57 | [demon] Выход из программы

INFO  | 2022-11-27 00:01:02,455  | file: demon_auto_purchase_bundle.py |  line: 44 | [demon] Поверяем истекающие премиум-пакеты

INFO  | 2022-11-27 00:01:02,480  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-27, количество людей с автопродлением подписки: 0

INFO  | 2022-11-27 00:01:02,483  | file: demon_auto_purchase_bundle.py |  line: 57 | [demon] Выход из программы



### **Решения**

#### **Задача 1**

Ваша задача написать функцию `count_success_and_failure`, которая принимает на вход путь к файлу с логами и подсчитывает количество успешных продлений и ошибок при списании. Функция должна вернуть кортеж из двух значений: количества успешных попыток и неуспешных.

**Решение**

Напишите свое решение ниже

In [25]:

def count_success_and_failure(file_path):
    attempts = 0
    failures = 0
    with open(file_path, "r") as file:
        for line in file:
            if line.startswith("ERROR"):
                failures += 1
    
            if "Обновляем подписку пользователю" in line:
                attempts += 1

    return attempts - failures, failures


count_success_and_failure('auto_purchase.log')

(1034, 186)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [26]:
res = count_success_and_failure('auto_purchase.log')

try:
    assert res == (1034, 186)
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача написать функцию `auto_renewal_sub`, которая принимает на вход путь к файлу с логами и обрабатывает количество клиентов с автопродлением подписки. Мы хотим посмотреть на изменение этого показателя в динамике: посчитайте сглаженные значения с помощью метода скользящего среднего и метода медианного сглаживания.  

**Примечание:** При сглаживании берем все предыдущие значения, включая текущее, будущие значения не берем. Если в один день наблюдаем несколько записей об автопродлении - берем максимальное из имеющихся число клиентов с подпиской.

Функция должна записать в файл `auto_renewal_sub.txt` два списка, предварив их соответствущими обозначениями:

`Среднее: [2.0, 1.0, 0.67...]`

`Медиана: [2, 2, 0...]`

**Решение**

Напишите свое решение ниже

In [88]:
def auto_renewal_sub(log_file_path):
    autorenewal_sub_info = {}
    with open(log_file_path, "r") as file:
        for line in file:
            status, datetime, filename, line_number, info = line.split(" | ")
            date_ = datetime.strip().split(" ")[0]
            info = info.strip()
            if "автопродлением подписки" in info:
                autorenewal_sub_number = int(info.split(" ")[-1])
                if date_ not in autorenewal_sub_info:
                    autorenewal_sub_info[date_] = set()
                autorenewal_sub_info[date_].add(autorenewal_sub_number)
    
    autorenewal_sub_numbers_mean = []
    autorenewal_sub_numbers_avg = []
    numbers_of_autorenewal = [max(i) for i in autorenewal_sub_info.values()]
    for index, _ in enumerate(numbers_of_autorenewal):
        current_windows = sorted(numbers_of_autorenewal[:index + 1])
        
        autorenewal_sub_numbers_avg.append(round(sum(current_windows) / len(current_windows), 2))

        if len(current_windows) % 2:
            centeral = current_windows[len(current_windows) // 2]
        else:
            left_number = current_windows[len(current_windows) // 2 - 1]
            right_number = current_windows[len(current_windows) // 2]
            centeral = (left_number + right_number) // 2

        autorenewal_sub_numbers_mean.append(centeral)
    
    with open("auto_renewal_sub.txt", "w") as file:
        file.write(f"Среднее: {autorenewal_sub_numbers_avg}")
        file.write(f"\nМедиана: {autorenewal_sub_numbers_mean}")
       

auto_renewal_sub('auto_purchase.log')



✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [86]:
import pandas as pd

user_answer = pd.read_csv('auto_renewal_sub.txt')
correct_answer = pd.read_csv('cor_auto_renewal.txt')

In [89]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

AssertionError: При проверке возникла ошибка AssertionError('Ответы не совпадают')

#### **Задача 3**

Напишите функцию `sub_renewal_by_day`, которая принимает на вход путь к файлу с логами и анализирует взаимосвязь дня продления подписки и количества продлений в этот день. Функция должна записать в файл `weekdays.txt` аналитическую записку в формате:

**`Количество обновлений подписки по дням недели:`**

**`Понедельник: 6`**

**`Вторник: 7`**

**`Среда: 8`**

**`...`**

**Решение**

Напишите свое решение ниже

In [44]:
def is_leap_year(year: int) -> bool:
    return (year % 4 == 0 and year % 100 != 0) or year % 400 == 0


def get_days_in_month(month: int, year: int) -> int:
    if month in (4, 6, 9, 11):
        return 30
    if month == 2:
        return 29 if is_leap_year(year) else 28
    return 31


def get_absolute_days(date_str: str) -> int:
    year, month, day = map(int, date_str.split("-"))

    days = day
    for m in range(1, month):
        days += get_days_in_month(m, year)
    return days

In [56]:
week = {}
def sub_renewal_by_day(file_path):
    with open(file_path, mode="r", encoding="utf-8") as file:
        for line in file:
            status, datetime, filename, number_of_line, info = line.strip().split(" | ")
            date = datetime.split(" ")[0]
            day_number = (get_absolute_days(date) + 4) % 7
            if status.strip() == "ERROR":
                continue
            if info.strip().startswith("[demon] Обновляем подписку пользователю id:"):
                week[day_number] = week.get(day_number, 0) + 1
week_names = {
    0: "Понедельник",
    1: "Вторник",
    2: "Среда",
    3: "Четверг",
    4: "Пятница",
    5: "Суббота",
    6: "Воскресенье"
}

sub_renewal_by_day('auto_purchase.log')

week = sorted(week.items())

for day_number, count in week:
    name = week_names[day_number]
    print(name, count)

Понедельник 160
Вторник 214
Среда 190
Четверг 172
Пятница 153
Суббота 170
Воскресенье 161


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [13]:
import pandas as pd

user_answer = pd.read_csv('weekdays.txt')
correct_answer = pd.read_csv('cor_weekdays.txt')

FileNotFoundError: [Errno 2] No such file or directory: 'weekdays.txt'

In [ ]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
